In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("../data/processed/telco_clean.csv")

print(df.shape)
print(df["Churn"].value_counts())

(7043, 21)
Churn
0    5174
1    1869
Name: count, dtype: int64


In [2]:
table=pd.crosstab(df["Contract"],df["Churn"])
print(table)

Churn              0     1
Contract                  
Month-to-month  2220  1655
One year        1307   166
Two year        1647    48


In [3]:
contract_table=(pd.crosstab(df["Contract"],df["Churn"],normalize='index'))*100
print(contract_table)

Churn                   0          1
Contract                            
Month-to-month  57.290323  42.709677
One year        88.730482  11.269518
Two year        97.168142   2.831858


In [4]:
categorical_columns = df.select_dtypes(include=["str"]).columns
categorical_columns = categorical_columns.drop("customerID")
categorical_columns = categorical_columns.tolist()
categorical_columns.append("SeniorCitizen")
print(categorical_columns)
print("Total features:", len(categorical_columns))

['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SeniorCitizen']
Total features: 16


In [5]:
results = []
for column in categorical_columns:
    table = pd.crosstab(df[column], df["Churn"])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.values.sum()
    rows, cols = table.shape
    cramers_v = np.sqrt(
        chi2 / (n * min(rows - 1, cols - 1))
    )
    results.append({
        "Feature": column,
        "Chi-square": chi2,
        "P-value": p,
        "Cramer's V": cramers_v,
        "Min Expected Count": expected.min()
    })
results_df = pd.DataFrame(results)
print(results_df.round(4))

             Feature  Chi-square  P-value  Cramer's V  Min Expected Count
0             gender      0.4841   0.4866      0.0083            925.6101
1            Partner    158.7334   0.0000      0.1501            902.7883
2         Dependents    189.1292   0.0000      0.1639            559.9304
3       PhoneService      0.9150   0.3388      0.0114            180.9823
4      MultipleLines     11.3304   0.0035      0.0401            180.9823
5    InternetService    732.3096   0.0000      0.3225            404.9544
6     OnlineSecurity    849.9990   0.0000      0.3474            404.9544
7       OnlineBackup    601.8128   0.0000      0.2923            404.9544
8   DeviceProtection    558.4194   0.0000      0.2816            404.9544
9        TechSupport    828.1971   0.0000      0.3429            404.9544
10       StreamingTV    374.2039   0.0000      0.2305            404.9544
11   StreamingMovies    375.6615   0.0000      0.2310            404.9544
12          Contract   1184.5966   0.0

In [6]:
print("Features with expected counts below 5:",(results_df["Min Expected Count"] < 5).sum())

Features with expected counts below 5: 0


In [7]:
results_df["Adjusted P-value"] = stats.false_discovery_control(results_df["P-value"],method="bh")
results_df["Significant"] = results_df["Adjusted P-value"] < 0.05
results_df["P-value display"] = results_df["P-value"].apply(lambda x: f"{x:.3e}")
results_df["Adjusted P-value display"] = results_df["Adjusted P-value"].apply(lambda x: f"{x:.3e}")
print(results_df[["Feature", "P-value", "Adjusted P-value", "Significant"]].round(5))

             Feature  P-value  Adjusted P-value  Significant
0             gender  0.48658           0.48658        False
1            Partner  0.00000           0.00000         True
2         Dependents  0.00000           0.00000         True
3       PhoneService  0.33878           0.36137        False
4      MultipleLines  0.00346           0.00396         True
5    InternetService  0.00000           0.00000         True
6     OnlineSecurity  0.00000           0.00000         True
7       OnlineBackup  0.00000           0.00000         True
8   DeviceProtection  0.00000           0.00000         True
9        TechSupport  0.00000           0.00000         True
10       StreamingTV  0.00000           0.00000         True
11   StreamingMovies  0.00000           0.00000         True
12          Contract  0.00000           0.00000         True
13  PaperlessBilling  0.00000           0.00000         True
14     PaymentMethod  0.00000           0.00000         True
15     SeniorCitizen  0.

In [8]:
retained = df[df["Churn"] == 0]["MonthlyCharges"]
churned = df[df["Churn"] == 1]["MonthlyCharges"]
print("Retained average:", retained.mean())
print("Churned average:", churned.mean())

Retained average: 61.26512369540008
Churned average: 74.44133226324237


In [9]:
t_stat, p_value = stats.ttest_ind(churned,retained,equal_var=False)
print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 18.407526676414673
P-value: 8.592449331547067e-73


In [10]:
n1 = len(churned)
n2 = len(retained)
pooled_std = np.sqrt(((n1 - 1) * churned.var() + (n2 - 1) * retained.var())/ (n1 + n2 - 2))
cohens_d = (churned.mean() - retained.mean()) / pooled_std
print("Cohen's d:", cohens_d)

Cohen's d: 0.446283496963266


In [11]:
monthly_charges_result = pd.DataFrame([{"Feature": "MonthlyCharges","Test": "Welch's t-test","Retained Mean": retained.mean(),"Churned Mean": churned.mean(),"T-statistic": t_stat,"P-value": p_value,"Cohens d": cohens_d}])
print(monthly_charges_result.round(4))

          Feature            Test  Retained Mean  Churned Mean  T-statistic  \
0  MonthlyCharges  Welch's t-test        61.2651       74.4413      18.4075   

   P-value  Cohens d  
0      0.0    0.4463  


In [12]:
from pathlib import Path
results_folder = Path("../reports/results")
results_folder.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_folder / "categorical_chi_square.csv",index=False)
monthly_charges_result.to_csv(results_folder / "monthly_charges_ttest.csv",index=False)

## Final Observations and Decisions
### 1. Key observations
- Contract has the strongest association with churn among the categorical features tested (Cramér's V = 0.410).
- OnlineSecurity (0.347), TechSupport (0.343), InternetService (0.323), and PaymentMethod (0.303) also show notable associations with churn.
- After Benjamini-Hochberg correction, 14 out of 16 categorical features remain statistically significant.
- Gender and PhoneService do not show statistically significant associations with churn.
- Churned customers have a higher average MonthlyCharges (74.44) than retained customers (61.27).
- Welch's t-test confirms a statistically significant difference (t = 18.408, p = 8.59e-73).
- Cohen's d = 0.446 indicates a small-to-moderate difference in monthly charges.

### 2. Statistical conclusions
- The chi-square tests indicate associations between several customer characteristics and churn.
- The Welch's t-test provides strong evidence that average MonthlyCharges differ between churned and retained customers.
- Expected frequencies were above 5 across all categorical tests, satisfying the usual expected-count guideline.
- These findings describe statistical associations and differences; they do not establish causation.

### 3. Decisions for the next stage
- Keep statistically significant categorical features as candidates for machine learning.
- Do not automatically remove Gender or PhoneService based only on these tests; their predictive contribution will be evaluated during modeling.
- Investigate Contract, OnlineSecurity, TechSupport, InternetService, and PaymentMethod further during model interpretation.
- Evaluate MonthlyCharges as a potentially useful numerical predictor.
- Use model performance and validation results, not statistical significance alone, to make final feature-selection decisions.

**Stage 3 complete:** Statistical hypothesis testing and effect-size analysis are complete. The findings will guide feature preparation and model development in the next stage.
